# 06 — Testing and Classic Interview Coding Problems

Dependency overrides for testing, mocking an external call, and the hands-on problems that show up in FastAPI interviews: an in-memory CRUD API, a health-check with real dependency checks, idempotent webhook handling, and simple rate limiting.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*httpx2.*")   # silence a harmless TestClient/httpx notice

from fastapi import FastAPI, Depends, HTTPException, Request, status
from fastapi.testclient import TestClient
from pydantic import BaseModel
import hashlib
import hmac
import time

## 1. Overriding dependencies in tests

`app.dependency_overrides[original] = replacement` swaps a dependency for the duration of testing — the standard way to replace a real DB/auth dependency with a fake one, without touching the route code at all. This is the actual FastAPI testing pattern, not something specific to these notebooks.

In [2]:
app = FastAPI()

def get_current_user():
    raise HTTPException(status_code=401, detail="real auth would check a token here")

@app.get("/whoami")
def whoami(user: dict = Depends(get_current_user)):
    return user

def fake_current_user():
    return {"user_id": 99, "username": "test-user"}

app.dependency_overrides[get_current_user] = fake_current_user

client = TestClient(app)
print(client.get("/whoami").json())   # bypasses real auth entirely, as intended in a test

app.dependency_overrides.clear()   # always clean up so overrides don't leak into other tests

{'user_id': 99, 'username': 'test-user'}


## 2. Classic problem: in-memory CRUD API

The single most common FastAPI interview exercise: full CRUD over an in-memory store, with correct status codes (`404` for a missing resource, `201` for creation) and `response_model` to shape output.

In [3]:
class TaskIn(BaseModel):
    title: str
    done: bool = False

class TaskOut(TaskIn):
    id: int

app2 = FastAPI()
TASKS: dict[int, TaskOut] = {}
_next_id = {"value": 1}

@app2.post("/tasks", response_model=TaskOut, status_code=status.HTTP_201_CREATED)
def create_task(task: TaskIn):
    task_id = _next_id["value"]
    _next_id["value"] += 1
    TASKS[task_id] = TaskOut(id=task_id, **task.model_dump())
    return TASKS[task_id]

@app2.get("/tasks/{task_id}", response_model=TaskOut)
def get_task(task_id: int):
    if task_id not in TASKS:
        raise HTTPException(status_code=404, detail="Task not found")
    return TASKS[task_id]

@app2.patch("/tasks/{task_id}", response_model=TaskOut)
def update_task(task_id: int, task: TaskIn):
    if task_id not in TASKS:
        raise HTTPException(status_code=404, detail="Task not found")
    TASKS[task_id] = TaskOut(id=task_id, **task.model_dump())
    return TASKS[task_id]

@app2.delete("/tasks/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_task(task_id: int):
    if task_id not in TASKS:
        raise HTTPException(status_code=404, detail="Task not found")
    del TASKS[task_id]

In [4]:
client2 = TestClient(app2)

created = client2.post("/tasks", json={"title": "write notebook"})
print(created.status_code, created.json())

task_id = created.json()["id"]
print(client2.get(f"/tasks/{task_id}").json())
print(client2.patch(f"/tasks/{task_id}", json={"title": "write notebook", "done": True}).json())
print(client2.delete(f"/tasks/{task_id}").status_code)
print(client2.get(f"/tasks/{task_id}").status_code)   # 404 -- confirmed deleted
print(client2.get("/tasks/9999").status_code)          # 404 -- never existed

201 {'title': 'write notebook', 'done': False, 'id': 1}
{'title': 'write notebook', 'done': False, 'id': 1}
{'title': 'write notebook', 'done': True, 'id': 1}
204
404
404


## 3. Classic problem: verified webhook receiver

"Accept a webhook, verify its HMAC signature, and process it **exactly once** even if the sender retries the same delivery." Two real security/correctness concerns in one endpoint: signature verification (reject anything not actually from the claimed sender) and idempotency (a webhook sender that doesn't get a prompt 200 will retry — processing the same event twice must be safe).

In [5]:
WEBHOOK_SECRET = b"shared-secret"
processed_event_ids: set[str] = set()

def verify_signature(payload: bytes, signature: str) -> bool:
    expected = hmac.new(WEBHOOK_SECRET, payload, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, signature)   # constant-time compare -- avoids timing attacks

@app2.post("/webhooks/payment")
async def receive_webhook(request: Request):
    payload = await request.body()
    signature = request.headers.get("x-signature", "")
    if not verify_signature(payload, signature):
        raise HTTPException(status_code=401, detail="invalid signature")

    body = await request.json()
    event_id = body["event_id"]
    if event_id in processed_event_ids:
        return {"status": "already_processed", "event_id": event_id}

    processed_event_ids.add(event_id)
    return {"status": "processed", "event_id": event_id}

import json as _json
body_bytes = _json.dumps({"event_id": "evt_1", "amount": 100}).encode()
good_sig = hmac.new(WEBHOOK_SECRET, body_bytes, hashlib.sha256).hexdigest()

print(client2.post("/webhooks/payment", content=body_bytes, headers={"x-signature": "bad-sig"}).status_code)
first = client2.post("/webhooks/payment", content=body_bytes, headers={"x-signature": good_sig})
print(first.json())
retry = client2.post("/webhooks/payment", content=body_bytes, headers={"x-signature": good_sig})
print(retry.json())   # same event_id, processed only once

401
{'status': 'processed', 'event_id': 'evt_1'}
{'status': 'already_processed', 'event_id': 'evt_1'}


## 4. Classic problem: simple rate limiting

A minimal fixed-window rate limiter as a dependency — enough to demonstrate the concept in an interview setting. (A production system would use Redis for shared state across multiple server processes/instances; this in-memory version only limits within one process, which is worth saying out loud if asked.)

In [6]:
request_log: dict[str, list[float]] = {}

def rate_limiter(request: Request, max_requests: int = 3, window_seconds: float = 1.0):
    client_id = request.client.host
    now = time.time()
    timestamps = [t for t in request_log.get(client_id, []) if now - t < window_seconds]
    if len(timestamps) >= max_requests:
        raise HTTPException(status_code=429, detail="rate limit exceeded")
    timestamps.append(now)
    request_log[client_id] = timestamps

@app2.get("/limited", dependencies=[Depends(rate_limiter)])
def limited_endpoint():
    return {"ok": True}

results = [client2.get("/limited").status_code for _ in range(5)]
print(results)   # first 3 succeed (200), rest hit the limit (429)

[200, 200, 200, 429, 429]


## Final interview checklist

- Can you explain what FastAPI is built on (Starlette + Pydantic) and why that combination matters?
- Can you write a CRUD API from memory with correct status codes (`201`, `404`, `204`)?
- Can you explain the `async def` blocking-call footgun and why FastAPI runs sync `def` routes in a thread pool?
- Can you design a dependency for auth, and one with `yield` for resource cleanup?
- Can you explain offset vs. cursor pagination and when each is appropriate?
- Can you implement HMAC signature verification and idempotent processing for a webhook?

**Related practice in this repo:** `../../pandas_practice/` and `../../spark_practice/` cover the data-processing side of a pipeline; this project covers the API layer that often sits in front of or behind one (ingestion endpoints, batch-prediction serving, exporting query results).